In [24]:
import numpy as np
from pathlib import Path
import h5py
from scipy.fft import  ifftshift, ifft


In [25]:
def workspace_creation(NN, MM, T_val):
    
    k_arr = np.array([-1,0,1])

    n  = np.arange(NN) - NN//2    

    n0 = np.linspace(0, 1, MM, endpoint=False)*NN - NN//2

    A_phase = np.exp(+1.j * k_arr[:, None, None]* 2 * np.pi * NN * np.arange(MM)/MM   )

    g  = np.exp(-(n[None,None,:] + k_arr[:,None,None]*NN
                        - n0[None,:,None] )**2 * 2 * T_val / 4 )

    N_cal = Ninv2(A_phase, g)

    return A_phase, g, N_cal


def Ninv2(A_phase, g):
    ph = A_phase[:, 0, :]
    gamma = np.einsum('kmn, smn -> ksm', g, g)
    return 1.0 / np.einsum('ksm, kj, sj -> mj', gamma, ph, ph.conj()).real


def Qfunc(psi, NN, MM, T_val, N_cal, Hp,  A_phase, g):

    Aq = 0
    for i in range(3):
        h = ifftshift(g[i] * psi[None, :], axes=-1)
        Hp[:, :NN//2] = h[:, :NN//2]
        Hp[:, -(NN//2):] = h[:, NN//2:]
        Aq = Aq + A_phase[i] * (MM * ifft(Hp, axis=-1, workers=-1, overwrite_x=True ))
    return N_cal * np.abs(Aq)**2 / (2*np.pi*T_val)

In [26]:

files = sorted(p.name for p in Path(r".\Data").glob("qsm*"))

names = np.array(files)

print(names)

['qsm_nq10_nint0_K1.41_LL1_iter1000.h5'
 'qsm_nq10_nint0_K1.41_LL67_iter1000.h5'
 'qsm_nq10_nint409_K-1.41_LL1_iter1000.h5']


In [27]:
"Select the name of the file"   

tmin0 = 500     

"Open the file and import the parameters and wavefunction"
with h5py.File(rf'.\Data\{names[2]}', 'r') as f:

    grp = f['dataset']

    nq = grp.attrs['nq']
    
    k_val = grp.attrs['k']
    
    T_val = grp.attrs['T']

    nint = grp.attrs['nint']

    K_val = k_val * T_val

    LL = grp.attrs['L']

    tmax = grp['psi'].shape[0]
    
    dim_H = grp['psi'].shape[1]


    ns = np.arange(dim_H) - dim_H //2

    MM = 2*dim_H

    APH, GG, Norm = workspace_creation(dim_H, MM, T_val)
    # print(APH.shape,  GG.shape)

    qmat = np.zeros((MM, MM), dtype=float)
    ncount = 0

    Hp_void = np.zeros((MM, MM), complex)                
    
    for indx in range(tmin0, tmax):
        
        psi = grp['psi'][indx,:]

        qmat += Qfunc(psi, dim_H, MM, T_val, N_cal=Norm, Hp=Hp_void, A_phase=APH, g=GG)
        ncount += 1
        
    qmat /= ncount


with h5py.File(rf'.\Data\Husimi_nq{nq:n}_nint{nint:n}_K{K_val:.2f}_LL{LL:n}_tmean{tmin0:n}_{tmax:n}.h5', 'w') as f:

    grp = f.create_group('dataset')

    grp.attrs['nq'] = nq
    grp.attrs['k'] = k_val
    grp.attrs['T'] = T_val
    grp.attrs['L'] = LL
    grp.attrs['nint'] = nint
    
    dst = grp.create_dataset('Qfunc', data = qmat, shape = (MM, MM), dtype='f8', compression='lzf')

